# Faster R-CNN: two-stage parameter experiments (Kaggle)

Runs the two extra experiments, each changing **one** parameter from the baseline
(`--rpn-proposals 2000`, `--roi-pos-fraction 0.25`):

| Run | Change | Parameter |
|---|---|---|
| `rpn4000` | `--rpn-proposals 4000` | number of RPN proposals (stage 1) |
| `pos0125` | `--roi-pos-fraction 0.125` | RoI sampling ratio (stage 2) |

Before running: **Settings -> Accelerator -> GPU**, **Internet on**, and add the competition data
(**Add Input -> multiple-object-video-detection-hw-2-ucu-ai-2026**).
Keep `EPOCHS`, `BATCH` and `SUBSET_STEP` equal to the baseline run, or the numbers are not comparable.
If one session (12 h) is too short for both, run with one name in `RUN`, then the other.

In [ ]:
!nvidia-smi
!git clone -q https://github.com/LiliAlania/drone-multi-object-detection.git /kaggle/working/repo
%cd /kaggle/working/repo
!pip install -q pycocotools

In [ ]:
EXPERIMENTS = {
    "rpn4000": "--rpn-proposals 4000",
    "pos0125": "--roi-pos-fraction 0.125",
}
RUN = ["rpn4000", "pos0125"]
EPOCHS = 12      # same as the baseline
BATCH = 4        # same as the baseline
SUBSET_STEP = 5  # same frame subsampling as the baseline
EVAL_STEP = 5    # evaluate every 5th val frame, as in the README

In [ ]:
from pathlib import Path

# find the competition split folders without walking every image
INPUT = Path("/kaggle/input")
found = [p for depth in range(1, 5) for p in INPUT.glob("*/" * depth + "val/annotations_coco.json")]
assert found, "Competition data not found: add it with Add Input"
VAL_ROOT = found[0].parent
DATA = VAL_ROOT.parent
print("data:", DATA, sorted(p.name for p in DATA.iterdir()))

SUBSET = Path("/kaggle/working/visdrone-subset")
if not (SUBSET / "images" / "train").is_dir():
    !python -m src.make_subset --data-root {DATA} --output-dir {SUBSET} --step {SUBSET_STEP} --splits train val
    !rm -f {SUBSET}.zip

In [ ]:
for name in RUN:
    flags = EXPERIMENTS[name]
    out = Path(f"/kaggle/working/runs/frcnn_{name}")
    resume = "--resume" if (out / "last.pt").is_file() else ""  # rerunning this cell continues an interrupted run
    print(f"=== {name}: {flags} ===")
    !python -m src.faster_rcnn train --data-root {SUBSET} --epochs {EPOCHS} --batch {BATCH} --workers 4 {flags} --out-dir {out} {resume}
    !python -m src.faster_rcnn evaluate --weights {out}/last.pt --split-root {VAL_ROOT} --batch {BATCH} --step {EVAL_STEP} | tee {out}/eval.txt

In [ ]:
# summary: mAP, AP50, AP_small and inference speed per run
for name in RUN:
    print(f"--- {name} ({EXPERIMENTS[name]})")
    !grep -aE "mAP@|Inference" /kaggle/working/runs/frcnn_{name}/eval.txt

In [ ]:
# keep only checkpoints and eval logs in the saved output
!rm -rf {SUBSET} /kaggle/working/repo